# Corrected executed small-scale adaptation: PowerMLP

**Original paper:** Ruichen Qiu, Yibo Miao, Shiwen Wang, and Yifan Zhu, *PowerMLP: An Efficient Version of KAN*, AAAI 39(19), 20069–20076 (2025). DOI: https://doi.org/10.1609/aaai.v39i19.34210.

This notebook implements the paper's published adapted layer formula for a small regression task: for every non-final layer, `z = alpha b(x) + ReLU_k(Wx + gamma)`, where `b(x)=x/(1+exp(-x))` and `ReLU_k(u)=max(0,u)^k`; the final layer is affine. The NumPy implementation is a small adaptation, not the paper's benchmark suite or a claim about its reported speed. The forward pass and analytic backpropagation are checked against numerical finite differences before training.


In [1]:
import json, time
from pathlib import Path
import numpy as np
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error

SEED=42; rng=np.random.default_rng(SEED)
x=np.linspace(-3,3,600).reshape(-1,1); y=(np.sin(2*x[:,0])+0.35*x[:,0]**2-0.15*x[:,0]**3).astype(float)
perm=rng.permutation(len(x)); tr=perm[:450]; te=perm[450:]; Xtr,Xte=x[tr],x[te]; ytr,yte=y[tr],y[te]

def basis(x):
    s=1.0/(1.0+np.exp(-x))
    return x*s

def basis_prime(x):
    s=1.0/(1.0+np.exp(-x))
    return s + x*s*(1.0-s)

def relu_k(x,k=2): return np.maximum(x,0.0)**k
def relu_k_prime(x,k=2): return k*np.maximum(x,0.0)**(k-1)*(x>0)

class PowerMLP:
    # Correct adapted layer: alpha*b(x) + ReLU_k(Wx + gamma), then final affine.
    def __init__(self, widths=(16,16), k=2, lr=0.01, epochs=1800):
        self.widths=widths; self.k=k; self.lr=lr; self.epochs=epochs

    def _initialize(self, input_dim):
        rng=np.random.default_rng(SEED); dims=[input_dim,*self.widths,1]
        self.W=[rng.normal(0,.35,(dims[i+1],dims[i])) for i in range(len(dims)-1)]
        self.b=[np.zeros(dims[i+1]) for i in range(len(dims)-1)]
        self.A=[rng.normal(0,.05,(dims[i+1],dims[i])) for i in range(len(dims)-2)]

    def _forward(self,X):
        xs=[X]; zs=[]
        for i in range(len(self.W)-1):
            z=xs[-1]@self.W[i].T+self.b[i]
            h=relu_k(z,self.k)+basis(xs[-1])@self.A[i].T
            zs.append(z); xs.append(h)
        out=xs[-1]@self.W[-1].T+self.b[-1]
        return xs,zs,out

    def _loss_and_gradients(self,X,y,need_input_grad=False):
        xs,zs,out=self._forward(X); n=len(X)
        err=out[:,0]-y; loss=float(np.mean(err**2)); d=2*err[:,None]/n
        gW=[None]*len(self.W); gb=[None]*len(self.b); gA=[None]*len(self.A)
        gW[-1]=d.T@xs[-1]; gb[-1]=d.sum(0); upstream=d@self.W[-1]
        for i in reversed(range(len(self.W)-1)):
            xprev=xs[i]
            dz=upstream*relu_k_prime(zs[i],self.k)
            gW[i]=dz.T@xprev; gb[i]=dz.sum(0); gA[i]=upstream.T@basis(xprev)
            upstream=dz@self.W[i] + (upstream@self.A[i])*basis_prime(xprev)
        return loss,(gW,gb,gA),upstream if need_input_grad else None

    def fit(self,X,y):
        self._initialize(X.shape[1])
        for _ in range(self.epochs):
            _,(gW,gb,gA),_=self._loss_and_gradients(X,y)
            for i in range(len(self.W)): self.W[i]-=self.lr*gW[i]; self.b[i]-=self.lr*gb[i]
            for i in range(len(self.A)): self.A[i]-=self.lr*gA[i]
        return self

    def predict(self,X): return self._forward(X)[2][:,0]

def finite_difference_check():
    model=PowerMLP(widths=(3,2),epochs=1); model._initialize(2)
    X=np.array([[-0.8,0.4],[0.6,-0.3]],dtype=float); target=np.array([0.2,-0.1])
    loss,(gW,gb,gA),gX=model._loss_and_gradients(X,target,need_input_grad=True)
    eps=1e-5
    def objective(): return model._loss_and_gradients(X,target)[0]
    max_param_error=0.0
    for arrays,grads in ((model.W,gW),(model.b,gb),(model.A,gA)):
        for arr,grad in zip(arrays,grads):
            for idx in np.ndindex(arr.shape):
                original=arr[idx]; arr[idx]=original+eps; plus=objective(); arr[idx]=original-eps; minus=objective(); arr[idx]=original
                max_param_error=max(max_param_error,abs((plus-minus)/(2*eps)-grad[idx]))
    numeric_X=np.zeros_like(X)
    for idx in np.ndindex(X.shape):
        original=X[idx]; X[idx]=original+eps; plus=objective(); X[idx]=original-eps; minus=objective(); X[idx]=original
        numeric_X[idx]=(plus-minus)/(2*eps)
    max_input_error=float(np.max(np.abs(numeric_X-gX)))
    print(json.dumps({"finite_difference_max_parameter_error":max_param_error,"finite_difference_max_input_error":max_input_error},indent=2))
    assert max_param_error < 1e-6 and max_input_error < 1e-6
    return max_param_error,max_input_error

finite_difference_check()

def evaluate(name, estimator):
    t=time.perf_counter(); estimator.fit(Xtr,ytr); elapsed=time.perf_counter()-t; pred=estimator.predict(Xte)
    return {"condition":name,"rmse":round(float(mean_squared_error(yte,pred)**0.5),6),"mae":round(float(mean_absolute_error(yte,pred)),6),"fit_seconds":round(float(elapsed),6),"n_train":len(Xtr),"n_test":len(Xte)}

results=[evaluate("two-hidden-layer ReLU MLP",MLPRegressor(hidden_layer_sizes=(16,16),solver="lbfgs",random_state=SEED,max_iter=500)),evaluate("corrected published-formula PowerMLP adaptation",PowerMLP())]
out={"task":"deterministic one-dimensional regression","formula":"z = alpha*b(x) + ReLU_k(Wx + gamma), b(x)=x/(1+exp(-x)), k=2; final layer affine","seed":SEED,"gradient_check":"central finite differences for all parameters and inputs before training","results":results,"limitation":"small NumPy adaptation, not the authors' benchmark or speed claim"}
Path("results.json").write_text(json.dumps(out,indent=2)); print(json.dumps(out,indent=2))


{
  "finite_difference_max_parameter_error": 9.537058642816731e-13,
  "finite_difference_max_input_error": 2.9552884662012446e-13
}
{
  "task": "deterministic one-dimensional regression",
  "formula": "z = alpha*b(x) + ReLU_k(Wx + gamma), b(x)=x/(1+exp(-x)), k=2; final layer affine",
  "seed": 42,
  "gradient_check": "central finite differences for all parameters and inputs before training",
  "results": [
    {
      "condition": "two-hidden-layer ReLU MLP",
      "rmse": 0.113449,
      "mae": 0.064928,
      "fit_seconds": 0.13278,
      "n_train": 450,
      "n_test": 150
    },
    {
      "condition": "corrected published-formula PowerMLP adaptation",
      "rmse": 0.334455,
      "mae": 0.292966,
      "fit_seconds": 2.491633,
      "n_train": 450,
      "n_test": 150
    }
  ],
  "limitation": "small NumPy adaptation, not the authors' benchmark or speed claim"
}
